# Analyse und Modell: Wetter und Heimvorteil

**Forschungsfrage:** Wie hängen Wetterbedingungen zum Anpfiff mit Spielergebnissen und dem Heimvorteil zusammen, und unterscheiden sich diese Zusammenhänge zwischen Premier League, Bundesliga und La Liga?

Dieses Notebook gehört zu Benjamins Teil (EDA, statistische Tests und Modellierung). Die Auswertung beschreibt Zusammenhänge; sie belegt keine Kausalität.


In [ ]:
# Falls nötig einmalig installieren: %pip install pandas matplotlib seaborn scipy scikit-learn
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, f_oneway, kruskal
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from sklearn.cluster import KMeans

pd.set_option("display.max_columns", 40)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
DATA_PATH = Path("data/spiele_roh_with_weather.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Datei fehlt: {DATA_PATH}. Lege die angereicherte CSV in den Ordner data/.")
df = pd.read_csv(DATA_PATH)
print(f"{df.shape[0]:,} Spiele, {df.shape[1]} Variablen")
df.head()

## 1. Datenprüfung und Vorbereitung

Die Ein-Spiel-pro-Zeile-Tabelle enthält Spielresultate, Liga/Saison, Anpfiff, Koordinaten und aggregierte Wetterwerte. Das eine am grünen Tisch entschiedene Spiel (`AWARDED`) wird wie vereinbart ausgeschlossen. Danach prüfen wir Schlüssel, Wertebereiche und fehlende Wetterwerte.


In [ ]:
# Ein Match pro Zeile?
assert df["match_id"].is_unique, "match_id ist nicht eindeutig"
print("Status:", df["status"].value_counts(dropna=False).to_dict())
print("Fehlende Werte (nur Spalten mit Lücken):")
print(df.isna().sum().loc[lambda s: s.gt(0)])

df["tore_heim"] = pd.to_numeric(df["tore_heim"], errors="coerce")
df["tore_gast"] = pd.to_numeric(df["tore_gast"], errors="coerce")
weather_cols = ["regen_summe", "temperatur_mittel", "wind_mittel", "snowfall_summe"]
for col in weather_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Gemäss Projektentscheid AWARDED-Spiele nicht in die Hauptanalyse aufnehmen.
analysis = df.loc[df["status"].eq("FINISHED")].copy()
analysis["tore_total"] = analysis["tore_heim"] + analysis["tore_gast"]
analysis["heim_sieg"] = analysis["resultat"].eq("H").astype(int)
analysis["temperatur_gruppe"] = pd.cut(analysis["temperatur_mittel"], bins=[-np.inf, 5, 15, np.inf], labels=["kalt (≤5°C)", "mild (5–15°C)", "warm (>15°C)"])
print(f"Hauptanalyse: {len(analysis):,} Spiele; ausgeschlossen: {len(df)-len(analysis)}")
print("Ergebnis-Codes:", analysis["resultat"].value_counts().to_dict())
analysis[weather_cols].describe().round(2)

## 2. Explorative Datenanalyse

Zuerst beschreiben wir den Heimvorteil und die Tore nach Liga und Saison. Die Wetterverteilungen zeigen, ob sich die Ligen klimatisch genug unterscheiden, um den Vergleich sinnvoll zu machen.


In [ ]:
home_by_league = analysis.groupby("liga", observed=True).agg(
    heim_siegquote=("heim_sieg", "mean"), spiele=("match_id", "count"),
    tore_pro_spiel=("tore_total", "mean")
).sort_values("heim_siegquote", ascending=False)
home_by_league.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.barplot(data=analysis, x="liga", y="heim_sieg", estimator="mean", errorbar=("ci", 95), ax=axes[0])
axes[0].set(title="Heimsiegquote nach Liga (95%-KI)", xlabel="Liga", ylabel="Anteil Heimsiege", ylim=(0, 0.65))
sns.boxplot(data=analysis, x="liga", y="temperatur_mittel", ax=axes[1])
axes[1].set(title="Temperatur am Spielort nach Liga", xlabel="Liga", ylabel="Durchschnittstemperatur (°C)")
plt.tight_layout()

In [ ]:
season_league = analysis.pivot_table(index="saison", columns="liga", values="heim_sieg", aggfunc="mean", observed=True)
plt.figure(figsize=(8, 3.5))
sns.heatmap(season_league, annot=True, fmt=".1%", cmap="YlGnBu", vmin=0.3, vmax=0.55)
plt.title("Heimsiegquote nach Liga und Saison")
plt.xlabel("Liga"); plt.ylabel("Saison"); plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for col, ax, label in zip(["regen_summe", "temperatur_mittel", "wind_mittel"], axes, ["Regen (mm)", "Temperatur (°C)", "Wind (km/h)"]):
    sns.histplot(data=analysis, x=col, hue="resultat", bins=25, stat="density", common_norm=False, element="step", ax=ax)
    ax.set_title(f"{label} nach Ergebnis")
plt.tight_layout()

## 3. Statistische Tests (mit p-Werten)

- **Chi-Quadrat:** Prüft, ob die Verteilung Heimsieg/Unentschieden/Auswärtssieg zwischen den Ligen unterschiedlich ist. Dazu berichten wir Cramérs V als Effektstärke.
- **ANOVA:** Prüft, ob sich die mittlere Temperatur zwischen den drei Ergebniskategorien unterscheidet. Der p-Wert allein sagt nichts über die Grösse des Effekts aus.
- Weil Regen stark schief verteilt sein kann, ergänzen wir dafür den nichtparametrischen Kruskal-Wallis-Test.


In [ ]:
# Liga × Spielergebnis
contingency = pd.crosstab(analysis["liga"], analysis["resultat"]).reindex(columns=["H", "D", "A"], fill_value=0)
chi2, p_chi, dof, expected = chi2_contingency(contingency)
n = contingency.to_numpy().sum()
cramers_v = np.sqrt(chi2 / (n * min(contingency.shape[0]-1, contingency.shape[1]-1)))
print(contingency)
print(f"Chi²({dof})={chi2:.2f}, p={p_chi:.4g}, Cramérs V={cramers_v:.3f}")

# Temperatur × Spielergebnis
groups_temp = [g["temperatur_mittel"].dropna().to_numpy() for _, g in analysis.groupby("resultat")]
f_temp, p_temp = f_oneway(*groups_temp)
print(f"ANOVA Temperatur: F={f_temp:.2f}, p={p_temp:.4g}")

# Regen × Spielergebnis (robuster gegenüber schiefer Verteilung)
groups_rain = [g["regen_summe"].dropna().to_numpy() for _, g in analysis.groupby("resultat")]
h_rain, p_rain = kruskal(*groups_rain)
print(f"Kruskal-Wallis Regen: H={h_rain:.2f}, p={p_rain:.4g}")

## 4. Multiklassen-Logit: Ergebnis H / D / A

Wir vergleichen einen **Basisfall** (Liga und Saison) mit einem Modell, das zusätzlich Regen, Temperatur und Wind berücksichtigt. Um Unterschiede zwischen den Ligen abzubilden, enthält das Wettermodell Wetter×Liga-Interaktionen. Die Daten werden stratifiziert in Trainings- und Testdaten aufgeteilt; bewertet werden Accuracy und Macro-F1. Der Vergleich ist kein Kausalnachweis.


In [ ]:
features_base = ["liga", "saison"]
weather_main = ["regen_summe", "temperatur_mittel", "wind_mittel"]

# Die Premier League dient als Referenzliga.
# Für Bundesliga und La Liga ergänzen wir Wetter×Liga-Interaktionen.
reference_league = "Premier League"
interaction_cols = []
for league in sorted(analysis["liga"].dropna().unique()):
    if league == reference_league:
        continue
    safe = league.lower().replace(" ", "_")
    for weather in weather_main:
        col = f"{weather}_x_{safe}"
        analysis[col] = analysis[weather] * analysis["liga"].eq(league).astype(int)
        interaction_cols.append(col)

features_weather = features_base + weather_main + interaction_cols
target = "resultat"

train_idx, test_idx = train_test_split(
    analysis.index,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=analysis[target],
)
y_train = analysis.loc[train_idx, target]
y_test = analysis.loc[test_idx, target]


def make_model(feature_cols):
    categorical = [col for col in feature_cols if analysis[col].dtype == "object"]
    numeric = [col for col in feature_cols if col not in categorical]
    prep = ColumnTransformer([
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical),
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), numeric),
    ])
    return Pipeline([
        ("prep", prep),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ])


models = {
    "Liga + Saison": (make_model(features_base), features_base),
    "Liga + Saison + Wetter×Liga": (make_model(features_weather), features_weather),
}
rows = []
for name, (model, cols) in models.items():
    model.fit(analysis.loc[train_idx, cols], y_train)
    pred = model.predict(analysis.loc[test_idx, cols])
    rows.append({
        "Modell": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Macro-F1": f1_score(y_test, pred, average="macro"),
    })
    if name == "Liga + Saison + Wetter×Liga":
        weather_model = model
        weather_predictions = pred

results = pd.DataFrame(rows).set_index("Modell")
results.round(3)

In [ ]:
# Majority-class reference and detailed metrics for the weather model
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(np.zeros((len(y_train), 1)), y_train)
dummy_pred = dummy.predict(np.zeros((len(y_test), 1)))
print(f"Mehrheits-Baseline: Accuracy={accuracy_score(y_test, dummy_pred):.3f}; Macro-F1={f1_score(y_test, dummy_pred, average='macro'):.3f}")
print(classification_report(y_test, weather_predictions, labels=["H", "D", "A"], zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, weather_predictions, labels=["H", "D", "A"], cmap="Blues")
plt.title("Testdaten: Wettermodell")
plt.tight_layout()

### Modell richtig interpretieren

Prüft, ob das Wettermodell im Testdatensatz Accuracy **und** Macro-F1 gegenüber dem Liga-Saison-Modell verbessert. Ein kleiner oder fehlender Unterschied bedeutet, dass Wetter in diesem Modell keine klare zusätzliche Prognoseinformation liefert. Eine geringe Gesamtgüte wäre angesichts fehlender Teamstärke-, Aufstellungs- und Quotenvariablen plausibel. Für die Präsentation zusätzlich die Koeffizienten/Richtung des Wettermodells prüfen und keine Kausalität behaupten.


## 5. K-Means: Wettertypen und Resultate

Als Zusatzanalyse gruppiert K-Means Spiele anhand von Regen, Temperatur und Wind in drei Wetterprofile. Die Variablen werden standardisiert, damit Temperatur und Wind wegen ihrer Einheiten nicht automatisch stärker gewichtet werden. Danach vergleichen wir die Ergebnisanteile je Cluster. Cluster sind explorativ und dürfen nicht als natürliche oder kausale Kategorien ausgegeben werden.


In [ ]:
cluster_cols = ["regen_summe", "temperatur_mittel", "wind_mittel"]
cluster_data = analysis[cluster_cols].copy()
cluster_data["regen_summe"] = np.log1p(cluster_data["regen_summe"].clip(lower=0))
cluster_scaled = StandardScaler().fit_transform(cluster_data)
km = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=20)
analysis["wetter_cluster"] = km.fit_predict(cluster_scaled)
cluster_profile = analysis.groupby("wetter_cluster")[cluster_cols].mean().round(2)
cluster_profile["spiele"] = analysis.groupby("wetter_cluster").size()
cluster_profile

In [ ]:
cluster_results = pd.crosstab(analysis["wetter_cluster"], analysis["resultat"], normalize="index").reindex(columns=["H", "D", "A"], fill_value=0)
cluster_results.plot(kind="bar", stacked=True, figsize=(8, 4), color=["#2b6cb0", "#718096", "#dd6b20"])
plt.title("Ergebnisanteile je Wettercluster")
plt.xlabel("Wettercluster"); plt.ylabel("Anteil"); plt.legend(title="Ergebnis", labels=["Heimsieg", "Unentschieden", "Auswärtssieg"])
plt.tight_layout()
cluster_results.round(3)

## 6. Schlussfolgerungen für die Forschungsfrage

Formuliere die Schlussfolgerung erst nach Ausführung des Notebooks anhand der ausgegebenen Zahlen:

1. Wie stark unterscheiden sich Heim- und Auswärtssiegquote nach Liga/Saison?
2. Zeigen Chi-Quadrat/ANOVA/Kruskal-Wallis statistische Evidenz, und wie gross ist der beobachtete Effekt?
3. Verbessert Wetter das Modell auf den Testdaten gegenüber Liga und Saison?
4. Welche Wetterprofile liefert K-Means, und sind die Resultatunterschiede zwischen Clustern gross genug, um praktisch relevant zu sein?

**Einschränkungen:** Beobachtungsdaten, wenige Ligen und Saisons, keine Kontrolle für Teamstärke/Spieler/Schiedsrichter/Zuschauer, Wetterwerte räumlich interpoliert und über ein Zeitfenster aggregiert. Der Datensatz ist nicht geeignet, kleine Wettereffekte kausal zu belegen.
